# Stage 14 - Batch inference (CPU, no cloud call)

Scores segments with the **current registered model** (Stage 12) and writes one prediction per segment: probabilities, decided labels and the **knowledge-bearing flag** that sends a segment to knowledge-card extraction. The report gives speed per 1,000 segments for the comparison with the LLM teacher (H2). Settings: `configs/inference.yaml`. Resumable.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[ml]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.inference import classify

report = classify("inference")
{k: report[k] for k in ("model_version", "scored", "knowledge_bearing_share", "segments_per_second", "seconds_per_1k", "cost_usd")}

In [ ]:
import pandas as pd

pd.Series(report["knowledge_types"], name="segments").to_frame().join(pd.Series(report["domains"], name="segments (domain)"), how="outer")

## Speed and cost against the teacher (H2)

The local model costs nothing per segment; the teacher's cost and latency come from run T (`results/T/metrics_validation.json`).

In [ ]:
import json
from krs.config import RESULTS_DIR

teacher = json.loads((RESULTS_DIR / "T" / "metrics_validation.json").read_text()) if (RESULTS_DIR / "T" / "metrics_validation.json").exists() else {}
pd.DataFrame({
    "local model": {"seconds per 1k": report["seconds_per_1k"], "cost per 1k ($)": 0.0},
    "LLM teacher": {"seconds per 1k": (teacher.get("latency_s_median") or float("nan")) * 1000, "cost per 1k ($)": teacher.get("cost_usd_per_1k")},
})

In [ ]:
from krs.inference import knowledge_bearing

flagged = list(knowledge_bearing(report["output"]))
len(flagged), flagged[0] if flagged else None   # input to knowledge-card extraction